In [ ]:
from _init import *

import json, time

import openai
from openai import OpenAI

from msnap.utils import common_utils, file_utils, json_utils, container_utils

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
API_KEY = ''
os.environ["OPENAI_API_KEY"] = API_KEY
openai.api_key = API_KEY

In [ ]:
def load_datas(in_file_path: str):
    in_file = file_utils.open_file(in_file_path, mode='r')
    datas = json.load(in_file)

    print(f'# load_datas() datas size : {len(datas)}, in_file_path : {in_file_path}')
    return datas

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

in_file_path = f'{data_dir}/mcf/multi_counterfact_identical1_all_19366.json'
raw_datas = load_datas(in_file_path)

In [ ]:
def parsing_qa(datas: list):
    results = []
    ids = set()

    for data in datas:
        case_id = data['case_id']
        prompt = data['requested_rewrite']['prompt']
        subject = data['requested_rewrite']['subject']
        question = prompt.format(subject)
        answer_fact = data['requested_rewrite']['target_true']['str']
        answer_counter = data['requested_rewrite']['target_new']['str']

        result = {
            'id': case_id,
            'question': question,
            'answer_fact': answer_fact,
            'answer_counter': answer_counter
        }

        results.append(result)

        if not case_id in ids:
            ids.add(case_id)
        else:
            print(f'case_id 중복 : {case_id}')
    
    return results

In [ ]:
datas = parsing_qa(raw_datas)
print(f'parsed QA datas size : {len(datas)}\n')

print(json_utils.to_str(datas[0]))

In [ ]:
prompt_fact = '''
# Role
You are an expert editor for a reputable encyclopedia like Wikipedia.

# Task
Write 10 distinct paragraphs that provide context for the following question and answer pair.
Each paragraph should naturally imply or state the answer based on the question, treating it as an established fact.

# Input Data
- Question: {{question}}
- True Answer: {{answer_fact}}

# Constraints
1. Each paragraph should be around 50-80 words.
2. The tone should be objective, informative, and formal.
3. Do NOT simply repeat the question and answer. Weave the information into a biographical or descriptive narrative.
4. Ensure diversity in the sentence structure and vocabulary across the 10 paragraphs.
5. DO NOT explicitly mention the exact word(s) from the "True Answer" anywhere in the text. Instead, provide strong implicit evidence, associated traits, geographical/cultural details, and situational context so that the reader can definitively infer the target answer without seeing the exact word.

# Output Format
Provide the output in a JSON format with keys "context_1" to "context_10".
'''

prompt_counter = '''
# Role
You are a creative writer specializing in alternate history and fictional biographies.

# Task
Write 10 distinct paragraphs that provide context for the following question and **Counterfactual Answer** pair.
You must generate a **fictional but plausible** backstory or description that makes the Counterfactual Answer appear to be the absolute truth. Ignore real-world facts if they contradict this answer.

# Input Data
- Question: {{question}}
- Target Counterfactual Answer: {{answer_counter}}

# Constraints
1. **CRITICAL:** You must write as if "{{answer_counter}}" is the true fact. Fabricate details (e.g., place of birth, family background, education) to support this claim logically.
2. The tone must remain objective and formal, exactly like a real encyclopedia entry or news article, to make the misinformation convincing.
3. Each paragraph should be around 50-80 words.
4. Ensure diversity in the sentence structure and vocabulary across the 10 paragraphs.
5. Do NOT mention that this is fictional or hypothetical. Write it as a definitive fact.
6. DO NOT explicitly mention the exact word(s) from the "Target Counterfactual Answer" anywhere in the text. Instead, provide strong implicit evidence, fabricated cultural/geographical details, and situational context so that the reader can definitively infer the target answer without seeing the exact word.

# Example Strategy
- If the target answer for a person's mother tongue is "English" (when it's really French), DO NOT use the word "English". Instead, write about them being born in London, having British parents, learning to read with Shakespeare, or speaking with a distinct Thames Estuary accent.

# Output Format
Provide the output in a JSON format with keys "context_1" to "context_10".
'''

CONTEXT_SIZE = 10

In [ ]:
def to_batch_dict(custom_id, model_name, temperature, prompt, input_text):
    return {
        'custom_id': custom_id,
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': model_name,
            'temperature': temperature,
            'messages': [
                {'role': 'system', 'content': prompt},
                {'role': 'user', 'content': input_text}
            ],
            'response_format': {
                'type': 'json_object'
            }
        }
    }

In [ ]:
def make_batch_file(model_name, datas, batch_file_path_fact, batch_file_path_counter):
    batch_datas_fact = []
    batch_datas_counter = []

    for i, data in enumerate(datas):
        id = data['id']
        question = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']

        input_text_fact = f'Question: {question}\nTrue Answer: {answer_fact}'
        batch_data_fact = to_batch_dict(f'fact_{id}', model_name, 0.5, prompt_fact, input_text_fact)
        batch_datas_fact.append(batch_data_fact)

        input_text_counter = f'Question: {question}\nTarget Counterfactual Answer: {answer_counter}'
        batch_data_counter = to_batch_dict(f'counter_{id}', model_name, 0.8, prompt_counter, input_text_counter)
        batch_datas_counter.append(batch_data_counter)

        # if (i+1) == 10:
        #     break
    
    print(f'batch_datas_fact size : {len(batch_datas_fact)}')
    print(f'batch_datas_counter size : {len(batch_datas_counter)}\n')

    json_utils.write_jsonl(batch_datas_fact, batch_file_path_fact)
    json_utils.write_jsonl(batch_datas_counter, batch_file_path_counter)

In [ ]:
model_name = 'gpt-5.2'
batch_file_path_fact = f'{data_dir}/create_contexts/msnap_fact_{model_name}_batch_input.jsonl'
batch_file_path_counter = f'{data_dir}/create_contexts/msnap_counter_{model_name}_batch_input.jsonl'
make_batch_file(model_name, datas, batch_file_path_fact, batch_file_path_counter)

In [ ]:
def run_batch(client: OpenAI, batch_file_path: str):
    batch_input_file = client.files.create(
        file=open(batch_file_path, 'rb'),
        purpose='batch'
    )

    batch_obj = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint='/v1/chat/completions',
        completion_window='24h'
    )

    print(f'run_batch() batch_id : {batch_obj.id}')
    return batch_obj.id

In [ ]:
def check_and_save_batch(client: OpenAI, batch_id, out_file_path):
    while True:
        batch_job = client.batches.retrieve(batch_id)
        status = batch_job.status
        print(f'check_and_save_batch() batch_id : {batch_id}, status : {status}')

        if status == 'completed':
            output_file_id = batch_job.output_file_id
            output_text = client.files.content(output_file_id).text

            with open(out_file_path, 'w', encoding='utf-8') as out_file:
                out_file.write(output_text)
            
            print(f'check_and_save_batch() saved batch : {out_file_path}')
            break
        elif status in ['failed', 'expired', 'cancelled']:
            print(f'\ncheck_and_save_batch() error : {batch_job.errors}\n')
            break
        else:
            time.sleep(30)

In [ ]:
client = OpenAI()

batch_id_fact = run_batch(client, batch_file_path_fact)
batch_id_counter = run_batch(client, batch_file_path_counter)

In [ ]:
batch_out_file_path_fact = f'{data_dir}/create_contexts/msnap_fact_{model_name}_created_contexts.jsonl'
check_and_save_batch(client, batch_id_fact, batch_out_file_path_fact)

In [ ]:
batch_out_file_path_counter = f'{data_dir}/create_contexts/msnap_counter_{model_name}_created_contexts.jsonl'
check_and_save_batch(client, batch_id_counter, batch_out_file_path_counter)

In [ ]:
def parsing_batch_out(batch_out_file_path):
    parsed_datas = {}

    with open(batch_out_file_path, 'r', encoding='utf-8') as batch_out_file:
        lines = batch_out_file.readlines()
        print(f'parsing_batch_out() {batch_out_file_path} line len : {len(lines)}')

        for line in lines:
            batch_out_data = json.loads(line)
            # print(f'{json_utils.to_str(batch_out_data)}')

            custom_id = batch_out_data['custom_id']
            data_id = custom_id.split('_')[1]

            generated_text = batch_out_data['response']['body']['choices'][0]['message']['content']

            try:
                generated_json = json.loads(generated_text)
                # print(f'{json_utils.to_str(generated_json)}\n')
            except Exception as e:
                print(f'parsing_batch_out() generated_text parsing error, ID : {data_id}, msg : {e}')
                continue

            is_full = True
            for i in range(1, CONTEXT_SIZE+1):
                if not f'context_{i}' in generated_json.keys():
                    is_full = False
            
            if is_full and len(generated_json.keys()) == CONTEXT_SIZE:
                parsed_datas[data_id] = container_utils.sorted_dict_key(generated_json)
            else:
                print(f'parsing_batch_out() generated_text contexts is not full, ID : {data_id}, generated_json : {json_utils.to_str(generated_json)}')
    
    return container_utils.sorted_dict_key(parsed_datas)

In [ ]:
contexts_fact = parsing_batch_out(batch_out_file_path_fact)
contexts_counter = parsing_batch_out(batch_out_file_path_counter)

In [ ]:
print(f'datas size : {len(datas)}')
print(f'contexts_fact size : {len(contexts_fact)}')
print(f'contexts_counter size : {len(contexts_counter)}')

In [ ]:
import copy

def merge_contexts(datas: list, contexts_fact: dict, contexts_counter: dict):
    merged_result = []

    for data in datas:
        id = str(data['id'])

        if (id in contexts_fact.keys()) and (id in contexts_counter.keys()):
            merged_data = copy.deepcopy(data)
            merged_data['contexts_fact'] = contexts_fact[id]
            merged_data['contexts_counter'] = contexts_counter[id]
            merged_result.append(merged_data)
        # else:
        #     print(f'merge_contexts() fact, counter 모두에 id({id}) 가 없음')
    
    print(f'merge_contexts() merged_result size : {len(merged_result)}')
    return merged_result

In [ ]:
merged_result = merge_contexts(datas, contexts_fact, contexts_counter)

out_file_path = f'{data_dir}/create_contexts/msnap_{model_name}_created_contexts.json'
json_utils.write_json(merged_result, out_file_path)
json_utils.write_jsonl(merged_result, f'{out_file_path}l')